# Mean-Variance Portfolio Optimization

This notebook demonstrates the full workflow: download prices, estimate CAPM returns and Ledoit-Wolf covariance, optimize a max-Sharpe portfolio, run a monthly rebalance backtest, inspect metrics, and save results.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.backtester import backtest_monthly_rebalance
from src.data_loader import download_price_data
from src.optimizer import optimize_portfolio
from src.utils import (
    pretty_print_backtest,
    pretty_print_results,
    save_backtest_result,
    save_optimization_result,
)

In [ ]:
tickers = [
    "UNH",
    "MPC",
    "BE",
    "VOO",
    "AFL",
    "JPM",
    "VTI",
    "GOOG",
    "TMUS",
    "AFRM",
    "CMG",
    "B",
    "NVDA",
    "YUM",
    "NTR",
    "KMI",
    "CBRE",
    "SHEL",
    "XEL",
    "SCHG",
    "IDXX",
    "GFI",
    "RBLX",
]

market_ticker = "SPY"
start_date = "2021-01-01"
end_date = "2026-01-01"
risk_free_rate = 0.02

In [ ]:
prices = download_price_data(
    tickers,
    start_date,
    end_date,
    save_path=PROJECT_ROOT / "data" / "prices.csv",
)
market_prices = download_price_data([market_ticker], start_date, end_date)

prices.tail()

In [ ]:
result = optimize_portfolio(
    prices,
    market_prices=market_prices,
    risk_free_rate=risk_free_rate,
    min_weight=0.02,
    max_weight=0.15,
)

pretty_print_results(result)

In [ ]:
import pandas as pd

weights = pd.Series(result.weights, name="weight").sort_values(ascending=False)
weights.to_frame().style.format("{:.2%}")

In [ ]:
weights_path = save_optimization_result(
    result,
    PROJECT_ROOT / "results" / "weights.json",
    metadata={
        "benchmark": market_ticker,
        "start_date": start_date,
        "end_date": end_date,
    },
)

weights_path

## Monthly Rebalance Backtest

In [ ]:
backtest = backtest_monthly_rebalance(
    prices,
    market_prices=market_prices,
    lookback_days=252,
    initial_capital=1.0,
    risk_free_rate=risk_free_rate,
    min_weight=0.02,
    max_weight=0.15,
    transaction_cost_bps=5.0,
    optimization_failure_policy="carry_forward",
)

pretty_print_backtest(backtest)

In [ ]:
backtest.equity_curve.plot(title="Monthly Rebalance Equity Curve", figsize=(10, 4));

In [ ]:
backtest.weights_history.tail().style.format("{:.2%}")

In [ ]:
backtest_path = save_backtest_result(
    backtest,
    PROJECT_ROOT / "results" / "backtest_results.json",
    metadata={
        "benchmark": market_ticker,
        "start_date": start_date,
        "end_date": end_date,
        "rebalance": "monthly",
    },
)

backtest_path